# Week 5 — Customer Segmentation with K-Means

**Theme:** Unsupervised learning II — clustering (k-means)

A mall wants to understand its customers so it can target promotions better.
Nobody has labeled customers as "budget shopper" or "big spender" — but if we
plot income vs. spending, natural groups might just... appear. That's what
**clustering** does: find groups of similar points with no labels given.

**K-Means algorithm, in one paragraph:** pick `k` random cluster centers ->
assign every point to its nearest center -> move each center to the average of
its assigned points -> repeat until centers stop moving.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

## 1. Create a synthetic customer dataset

We simulate 5 realistic customer segments (this also means we secretly *know*
the "true" answer, which is useful for checking whether K-Means finds it).

In [ ]:
rng = np.random.default_rng(42)

segments = [
    # (mean_income_k$, mean_spending_score, n_customers)
    (25, 20, 40),   # low income, low spending
    (25, 80, 40),   # low income, high spending (impulsive)
    (55, 50, 40),   # mid income, mid spending
    (85, 20, 40),   # high income, low spending (frugal)
    (85, 85, 40),   # high income, high spending
]

incomes, spending = [], []
for mean_income, mean_spend, n in segments:
    incomes.append(rng.normal(mean_income, 5, n))
    spending.append(rng.normal(mean_spend, 8, n))

income = np.concatenate(incomes)
spending_score = np.concatenate(spending)
X = np.column_stack([income, spending_score])
print("Customers:", X.shape[0])

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], alpha=0.6, edgecolor="k")
plt.title("Customers: Annual Income vs. Spending Score (unlabeled)")
plt.xlabel("Annual income ($k)")
plt.ylabel("Spending score (1-100)")
plt.show()

## 2. How many clusters? The elbow method

We don't know `k` in advance. We try several values and plot "inertia" (how
tightly packed each cluster is) — look for the "elbow" where adding more
clusters stops helping much.

In [ ]:
inertias = []
k_range = range(1, 10)
for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X)
    inertias.append(km.inertia_)

plt.figure(figsize=(6, 4))
plt.plot(list(k_range), inertias, marker="o")
plt.title("Elbow Method")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia (within-cluster sum of squares)")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Silhouette score is another way to pick k: higher is better (max 1.0)
for k in range(2, 8):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    score = silhouette_score(X, km.labels_)
    print(f"k={k}: silhouette score = {score:.3f}")

## 3. Run K-Means with the chosen k

Both the elbow plot and the silhouette scores should point toward **k=5** —
which matches how we generated the data.

In [ ]:
k = 5
kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
cluster_labels = kmeans.fit_predict(X)

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c=cluster_labels, cmap="tab10", alpha=0.7, edgecolor="k")
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            c="red", marker="X", s=250, edgecolor="black", label="cluster center")
plt.title(f"K-Means Clustering (k={k})")
plt.xlabel("Annual income ($k)")
plt.ylabel("Spending score (1-100)")
plt.legend()
plt.show()

## 4. Interpret the segments

Turn the raw cluster centers into a business-readable table.

In [ ]:
import pandas as pd

summary = pd.DataFrame(kmeans.cluster_centers_, columns=["avg_income_k$", "avg_spending_score"])
summary["n_customers"] = pd.Series(cluster_labels).value_counts().sort_index().values
summary.index.name = "cluster"
summary

## Try it yourself

1. **Pick the wrong k.** Re-run K-Means with `k=2` and `k=8` and re-plot — how
   does the clustering change? Which one looks "wrong" given the elbow plot?
2. **Name the segments.** Based on the summary table, write a one-word label
   for each cluster (e.g. "frugal high earners", "impulsive low earners").
3. **Add a third feature.** Simulate a `visits_per_month` column and re-run
   K-Means on all 3 features — plot 2 of the 3 dimensions to visualize.
4. **Compare to Week 4.** Both PCA and K-Means are unsupervised — what's the
   key difference in what each one is *for*? (Hint: one compresses features,
   the other groups examples.)